<a href="https://colab.research.google.com/github/Tar-ive/dl_basics/blob/main/Linear_Regression_PyTorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Implement Linear Regression In PyTorch

In [30]:
# Imports
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
from torch.utils.data import DataLoader, Dataset

In [31]:
#Generate Synthetic Data
torch.manual_seed(42)
X = torch.rand(100,1) * 10 # generates random values (100 of them) from 0-10
y = 2 * X + 3 + torch.randn(100,1) # Linear Relationship with the Noise

#Save data to Csv File
data = torch.cat((X,y), dim = 1)
df = pd.DataFrame(data.numpy(), columns = ["X", "y"])
df.to_csv("data.csv", index = False)

In [32]:
# Write the Custom Dataset Class
class LinearRegressionDataset(Dataset):
  def __init__(self, csv_file):
    # Load data from csv file
    self.data = pd.read_csv(csv_file)
    self.X = torch.tensor(self.data["X"].values, dtype=torch.float32).view(-1,1 )
    self.y = torch.tensor(self.data["y"].values, dtype=torch.float32).view(-1,1 )

  def __len__(self):
    return len(self.data)

  def __getitem__(self, idx):
    return self.X[idx], self.y[idx]


#Example usage of DataLoader
dataset = LinearRegressionDataset("data.csv")
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

In [ ]:
# Define Linear Regression Model
class LinearRegressionModel(nn.Module):
  def __init__(self):
    super().__init__() # calls base class nn.Module
    self.linear = nn.Linear(1,1)

# Define Forward Pass

  def forward(self, X):
    return self.linear(X)

In [ ]:
#Initialize Model, Loss Function (MSELoss) and Optimizer (SGD)
model = LinearRegressionModel()
criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

In [33]:
#Training Loop
epochs = 1000
for epoch in range(epochs):
  for batch_X, batch_y in dataloader:
    #Forward pass
    predictions = model(batch_X)
    loss = criterion(predictions, batch_y)

    #Backward Pass and Optimization
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

  #Log Progress every 100 epochs
  if (epoch+1) % 100 == 0:
    print(f"Epoch:[{epoch+1}/{epochs}], loss:{loss.item():.4f}")

Epoch:[100/1000], loss:1.7103
Epoch:[200/1000], loss:0.1732
Epoch:[300/1000], loss:0.2909
Epoch:[400/1000], loss:0.9223
Epoch:[500/1000], loss:0.2024
Epoch:[600/1000], loss:1.0876
Epoch:[700/1000], loss:0.3212
Epoch:[800/1000], loss:0.4629
Epoch:[900/1000], loss:0.4640
Epoch:[1000/1000], loss:0.2040


In [34]:
#Display the Learned Parameters
[W, b]= model.linear.parameters()
print(f"Learned weight: {W.item():.4f}, Learned Bias:{b.item():.4f}")

Learned weight: 1.9603, Learned Bias:3.1974


In [35]:
#Testing on New Data
X_test = torch.tensor([[4.0], [7.0]])
with torch.no_grad():
  predictions = model(X_test)
  print(f"Predictions for {X_test.tolist()}:{predictions.tolist()}")

Predictions for [[4.0], [7.0]]:[[11.038619041442871], [16.919504165649414]]
